# Style Transfer Trade-Off Analysis

This notebook analyses the paper-scale style-transfer sweep generated by `sweep_mpc_flowchef.json`.

The style terminal loss is the squared distance between CLIP ViT-B/16 Gram-matrix style features of the decoded sample and the reference style image. The sweep fixes 20 conditioning updates per time step and varies the MPC control regularisation weight `rho` against the FlowChef terminal-loss learning rate.

The analysis summarises the trade-off between style adherence and content preservation across five prompts and nine style images. It also includes qualitative grid cells for selected prompt/style pairs.

Run from the `FLUX2/` directory after sweep outputs exist under `output/sweep/`. Figure files are written to `output/analysis/style/`.


In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

def _find_flux2_root(start: Path):
    for parent in [start] + list(start.parents):
        if (parent / 'cli_mpc_flow.py').exists() and (parent / 'sweeps').exists():
            return parent
    return None

repo_root = _find_flux2_root(Path.cwd().resolve()) or Path.cwd().resolve()
root = repo_root / 'output' / 'sweep'
analysis_dir = repo_root / 'output' / 'analysis' / 'style'
analysis_dir.mkdir(parents=True, exist_ok=True)
rows = []

def parse_token(path_str: str, prefix: str):
    m = re.search(rf'{prefix}=([^\\/]+)', path_str)
    return m.group(1) if m else None

for csv_path in root.rglob('*.csv'):
    path_str = str(csv_path)
    parts = csv_path.parts
    try:
        method = parts[parts.index('sweep') + 1]
    except ValueError:
        method = None
    style = parse_token(path_str, 'style')
    lr = parse_token(path_str, 'lr')
    its = parse_token(path_str, 'its')
    rho = parse_token(path_str, 'rho')
    prompt = csv_path.stem
    df = pd.read_csv(csv_path)
    if df.empty:
        continue
    row = df.iloc[0].to_dict()
    row.update({
        'method': method,
        'style': style,
        'lr': lr,
        'its': its,
        'rho': rho,
        'prompt': prompt,
        'path': path_str,
    })
    rows.append(row)

data = pd.DataFrame(rows)
data['style_loss'] = pd.to_numeric(data.get('style_loss'), errors='coerce')
data['avg_path_deviation'] = pd.to_numeric(data.get('avg_path_deviation'), errors='coerce')
data['clip_score'] = pd.to_numeric(data.get('clip_score'), errors='coerce')
data['dino_style_cosine'] = pd.to_numeric(data.get('dino_style_cosine'), errors='coerce')
data['lpips_to_original'] = pd.to_numeric(data.get('lpips_to_original'), errors='coerce')
data['its'] = pd.to_numeric(data.get('its'), errors='coerce')
data




## Optional Style/Content Metrics

The paper trade-off plot uses DINOv2 cosine similarity to the style image as the style-adherence metric and LPIPS to the unconditioned image as the content-preservation distance. If these columns are absent from the sweep CSVs, the next cell computes them in memory from the generated PNGs.


In [ ]:
import numpy as np
from PIL import Image

need_dino = "dino_style_cosine" not in data or data["dino_style_cosine"].isna().any()
need_lpips = "lpips_to_original" not in data or data["lpips_to_original"].isna().any()

if need_dino or need_lpips:
    import torch
    import torchvision.transforms as T

    device = "cuda" if torch.cuda.is_available() else "cpu"

    dino_model = None
    if need_dino:
        try:
            dino_model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14").to(device).eval()
            dino_preprocess = T.Compose([
                T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
                T.CenterCrop(224),
                T.ToTensor(),
                T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ])
        except Exception as exc:
            print(f"DINOv2 unavailable; dino_style_cosine will stay NaN: {exc}")
            dino_model = None

    lpips_model = None
    if need_lpips:
        try:
            import lpips
            lpips_model = lpips.LPIPS(net="vgg").to(device).eval()
        except Exception as exc:
            print(f"LPIPS unavailable; lpips_to_original will stay NaN: {exc}")
            lpips_model = None

    def _encode_dino(img: Image.Image):
        with torch.no_grad():
            tensor = dino_preprocess(img.convert("RGB")).unsqueeze(0).to(device)
            emb = dino_model(tensor)
            emb = emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-12)
            return emb.squeeze(0).cpu().numpy()

    def _to_lpips_tensor(img: Image.Image):
        arr = np.asarray(img.convert("RGB")).astype(np.float32) / 255.0
        arr = arr * 2.0 - 1.0
        return torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(device)

    def _image_path_from_csv(csv_path):
        return Path(csv_path).with_suffix(".png")

    style_emb_cache = {}
    if dino_model is not None:
        style_root = repo_root / "style_images"
        for img_path in list(style_root.glob("*.png")) + list(style_root.glob("*.jpg")) + list(style_root.glob("*.jpeg")):
            style_emb_cache[img_path.stem.lower()] = _encode_dino(Image.open(img_path).convert("RGB"))

    original_img_cache = {}
    original_rows = data[(data["method"] == "mpc") & (data["rho"].astype(str).str.lower() == "na")]
    for row in original_rows.itertuples():
        img_path = _image_path_from_csv(row.path)
        if img_path.exists():
            original_img_cache[(str(row.style).lower(), str(row.prompt).lower())] = img_path

    dino_values = []
    lpips_values = []
    for row in data.itertuples():
        img_path = _image_path_from_csv(row.path)
        if not img_path.exists():
            dino_values.append(np.nan)
            lpips_values.append(np.nan)
            continue

        img = Image.open(img_path).convert("RGB")

        dino_val = np.nan
        if dino_model is not None:
            style_emb = style_emb_cache.get(str(row.style).lower())
            if style_emb is not None:
                emb = _encode_dino(img)
                dino_val = float((emb * style_emb).sum())
        dino_values.append(dino_val)

        lpips_val = np.nan
        if lpips_model is not None:
            original_path = original_img_cache.get((str(row.style).lower(), str(row.prompt).lower()))
            if original_path is not None and original_path.exists():
                original_img = Image.open(original_path).convert("RGB")
                with torch.no_grad():
                    lpips_val = float(lpips_model(_to_lpips_tensor(img), _to_lpips_tensor(original_img)).item())
        lpips_values.append(lpips_val)

    if need_dino:
        data["dino_style_cosine"] = dino_values
    if need_lpips:
        data["lpips_to_original"] = lpips_values

for col in ["dino_style_cosine", "lpips_to_original"]:
    data[col] = pd.to_numeric(data.get(col), errors="coerce")

data[["method", "style", "prompt", "dino_style_cosine", "lpips_to_original"]].head()


In [ ]:
mpc = data[data['method'] == 'mpc'].copy()
mpc['rho_str'] = mpc['rho'].fillna('na').astype(str)
mpc['rho_val'] = pd.to_numeric(mpc['rho'], errors='coerce')
mpc = mpc.dropna(subset=['style_loss', 'avg_path_deviation', 'clip_score', 'its'])

mpc_group = (
    mpc.groupby(['rho_str', 'rho_val', 'its', 'prompt', 'style'], dropna=False)
    .agg(
        style_loss_mean=('style_loss', 'mean'),
        path_dev_mean=('avg_path_deviation', 'mean'),
        clip_score_mean=('clip_score', 'mean'),
        dino_style_cosine_mean=('dino_style_cosine', 'mean'),
        lpips_to_original_mean=('lpips_to_original', 'mean'),
    )
    .reset_index()
)
mpc_group = (
    mpc_group.groupby(['rho_str', 'rho_val', 'its'], dropna=False)
    .agg(
        style_loss_mean=('style_loss_mean', 'mean'),
        path_dev_mean=('path_dev_mean', 'mean'),
        clip_score_mean=('clip_score_mean', 'mean'),
        dino_style_cosine_mean=('dino_style_cosine_mean', 'mean'),
        lpips_to_original_mean=('lpips_to_original_mean', 'mean'),
        dino_style_cosine_std=('dino_style_cosine_mean', 'std'),
        lpips_to_original_std=('lpips_to_original_mean', 'std'),
        n=('style_loss_mean', 'count'),
    )
    .reset_index()
    .sort_values(['its', 'rho_val'])
)
mpc_group





In [ ]:
flow = data[data['method'] == 'flowchef'].copy()
flow = flow[flow['its'] != 0]
flow['lr'] = pd.to_numeric(flow['lr'], errors='coerce')
flow = flow.dropna(subset=['lr', 'style_loss', 'avg_path_deviation', 'clip_score', 'its'])

flow_group = (
    flow.groupby(['lr', 'its', 'prompt', 'style'], dropna=True)
    .agg(
        style_loss_mean=('style_loss', 'mean'),
        path_dev_mean=('avg_path_deviation', 'mean'),
        clip_score_mean=('clip_score', 'mean'),
        dino_style_cosine_mean=('dino_style_cosine', 'mean'),
        lpips_to_original_mean=('lpips_to_original', 'mean'),
    )
    .reset_index()
)
flow_group = (
    flow_group.groupby(['lr', 'its'], dropna=True)
    .agg(
        style_loss_mean=('style_loss_mean', 'mean'),
        path_dev_mean=('path_dev_mean', 'mean'),
        clip_score_mean=('clip_score_mean', 'mean'),
        dino_style_cosine_mean=('dino_style_cosine_mean', 'mean'),
        lpips_to_original_mean=('lpips_to_original_mean', 'mean'),
        dino_style_cosine_std=('dino_style_cosine_mean', 'std'),
        lpips_to_original_std=('lpips_to_original_mean', 'std'),
        n=('style_loss_mean', 'count'),
    )
    .reset_index()
    .sort_values(['its', 'lr'])
)
flow_group





In [ ]:
plt.figure(figsize=(8, 5))
for its in sorted(mpc_group['its'].unique()):
    subset = mpc_group[mpc_group['its'] == its]
    label = f'MPC its={int(its)} (by rho)'
    plt.scatter(subset['style_loss_mean'], subset['path_dev_mean'], label=label)
    for _, row in subset.iterrows():
        plt.text(row['style_loss_mean'], row['path_dev_mean'], 'rho=' + row['rho_str'])

for its in sorted(flow_group['its'].unique()):
    subset = flow_group[flow_group['its'] == its]
    label = f'FlowChef its={int(its)} (by lr)'
    plt.scatter(subset['style_loss_mean'], subset['path_dev_mean'], label=label, marker='x')
    for _, row in subset.iterrows():
        plt.text(row['style_loss_mean'], row['path_dev_mean'], 'lr=' + str(row['lr']))

plt.xlabel('Average style loss')
plt.ylabel('Average path deviation')
plt.title('MPC vs FlowChef: style loss vs path deviation')
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
for its in sorted(mpc_group['its'].unique()):
    subset = mpc_group[mpc_group['its'] == its]
    label = f'MPC its={int(its)} (by rho)'
    plt.scatter(subset['clip_score_mean'], subset['style_loss_mean'], label=label)
    for _, row in subset.iterrows():
        plt.text(row['clip_score_mean'], row['style_loss_mean'], 'rho=' + row['rho_str'])

for its in sorted(flow_group['its'].unique()):
    subset = flow_group[flow_group['its'] == its]
    label = f'FlowChef its={int(its)} (by lr)'
    plt.scatter(subset['clip_score_mean'], subset['style_loss_mean'], label=label, marker='x')
    for _, row in subset.iterrows():
        plt.text(row['clip_score_mean'], row['style_loss_mean'], 'lr=' + str(row['lr']))

plt.xlabel('Average CLIP agreement')
plt.ylabel('Average style loss')
plt.title('Style loss vs CLIP agreement')
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
for its in sorted(mpc_group['its'].unique()):
    subset = mpc_group[mpc_group['its'] == its]
    label = f'MPC its={int(its)} (by rho)'
    plt.scatter(subset['path_dev_mean'], subset['clip_score_mean'], label=label)
    for _, row in subset.iterrows():
        plt.text(row['path_dev_mean'], row['clip_score_mean'], 'rho=' + row['rho_str'])

for its in sorted(flow_group['its'].unique()):
    subset = flow_group[flow_group['its'] == its]
    label = f'FlowChef its={int(its)} (by lr)'
    plt.scatter(subset['path_dev_mean'], subset['clip_score_mean'], label=label, marker='x')
    for _, row in subset.iterrows():
        plt.text(row['path_dev_mean'], row['clip_score_mean'], 'lr=' + str(row['lr']))

plt.xlabel('Average path deviation')
plt.ylabel('Average CLIP agreement')
plt.title('Path deviation vs CLIP agreement')
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
for its in sorted(mpc_group['its'].unique()):
    subset = mpc_group[mpc_group['its'] == its]
    label = f'MPC its={int(its)} (by rho)'
    plt.scatter(subset['dino_style_cosine_mean'], subset['clip_score_mean'], label=label)
    for _, row in subset.iterrows():
        plt.text(row['dino_style_cosine_mean'], row['clip_score_mean'], 'rho=' + row['rho_str'])

for its in sorted(flow_group['its'].unique()):
    subset = flow_group[flow_group['its'] == its]
    label = f'FlowChef its={int(its)} (by lr)'
    plt.scatter(subset['dino_style_cosine_mean'], subset['clip_score_mean'], label=label, marker='x')
    for _, row in subset.iterrows():
        plt.text(row['dino_style_cosine_mean'], row['clip_score_mean'], 'lr=' + str(row['lr']))

plt.xlabel('Average DINO style cosine')
plt.ylabel('Average CLIP agreement')
plt.title('CLIP agreement vs DINO style cosine')
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 14,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120,
})

fig, ax = plt.subplots(figsize=(6.0, 4.0), constrained_layout=True)

colors = {
    "mpc": "#3e89cf",
    "flow": "#d4452c",
    "orig": "#666666",
}

def convert_sim_to_distance(sim_series):
    return 1 - sim_series

def convert_distance_to_sim(dist_series):
    return 1 - dist_series

# --- MPC ---
mpc_label_used = False
for its in sorted(mpc_group["its"].unique()):
    subset = mpc_group[mpc_group["its"] == its].sort_values("rho_val")
    subset = subset[subset["rho_str"] != "na"]
    if subset.empty:
        continue

    x = subset["dino_style_cosine_mean"].dropna()
    y = convert_distance_to_sim(subset["lpips_to_original_mean"]).dropna()

    print(x)
    print(y)

    ax.plot(x, y, linestyle=":", linewidth=1.0, alpha=0.5, color=colors["mpc"], zorder=1, label="_nolegend_")
    ax.plot(
        x, y, linestyle="None", marker="P", markersize=10,
        markeredgewidth=0.8, markeredgecolor="white",
        color=colors["mpc"], zorder=3,
        label="MPC-Flow" if not mpc_label_used else "_nolegend_",
    )
    mpc_label_used = True

# --- FlowChef ---
flow_label_used = False
for its in sorted(flow_group["its"].unique()):
    subset = flow_group[flow_group["its"] == its].sort_values("lr")
    if subset.empty:
        continue

    x = subset["dino_style_cosine_mean"]
    y = convert_distance_to_sim(subset["lpips_to_original_mean"])

    ax.plot(x, y, linestyle=":", linewidth=1.0, alpha=0.5, color=colors["flow"], zorder=1, label="_nolegend_")
    ax.plot(
        x, y, linestyle="None", marker="x", markersize=10,
        markeredgewidth=1.4,
        color=colors["flow"], zorder=3,
        label="FlowChef" if not flow_label_used else "_nolegend_",
    )
    flow_label_used = True

# --- Original ---
orig = mpc_group[mpc_group["rho_str"] == "na"]
if not orig.empty:
    ax.plot(
        (orig["dino_style_cosine_mean"]),
        (convert_distance_to_sim(orig["lpips_to_original_mean"])),
        linestyle="None", marker="s", markersize=10,
        markeredgewidth=0.8, markeredgecolor="white",
        color=colors["orig"], zorder=4,
        label="Unconditional",
    )

ax.set_xlabel("Style similarity \n(DINO cosine similarity to style) →")
ax.set_ylabel("Content similarity \n(1 - LPIPS to original) →")
ax.set_title("Style–content trade-off")

# Stable annotation placement
# ax.annotate(
#     "Better images",
#     xy=(0.80, 0.25), xytext=(0.90, 0.35),
#     xycoords="axes fraction", textcoords="axes fraction",
#     arrowprops=dict(arrowstyle="->", lw=1.2, color="black"),
#     ha="left", va="bottom",
# )

ax.grid(True, which="major", alpha=0.15, linewidth=0.8)
ax.set_xlim([-0.05, 0.4])

leg = ax.legend(frameon=False, loc="best", handletextpad=0.6)

fig.savefig(analysis_dir / "style_content_tradeoff.pdf", bbox_inches="tight")
fig.savefig(analysis_dir / "style_content_tradeoff.svg", bbox_inches="tight")
fig.savefig(analysis_dir / "style_content_tradeoff.png", dpi=400, bbox_inches="tight")
plt.show()


In [ ]:
x = subset["dino_style_cosine_mean"]
y = convert_distance_to_sim(subset["lpips_to_original_mean"])
x,y

In [ ]:
mpc_ps = (
    mpc.groupby(['prompt', 'style', 'its', 'rho_str'], dropna=False)
    .agg(
        style_loss_mean=('style_loss', 'mean'),
        clip_score_mean=('clip_score', 'mean'),
        dino_style_cosine_mean=('dino_style_cosine', 'mean'),
        lpips_to_original_mean=('lpips_to_original', 'mean'),
    )
    .reset_index()
)

flow_ps = (
    flow.groupby(['prompt', 'style', 'its', 'lr'], dropna=True)
    .agg(
        style_loss_mean=('style_loss', 'mean'),
        clip_score_mean=('clip_score', 'mean'),
        dino_style_cosine_mean=('dino_style_cosine', 'mean'),
        lpips_to_original_mean=('lpips_to_original', 'mean'),
    )
    .reset_index()
)

def format_lr(val):
    try:
        return f"{float(val):.3g}"
    except (TypeError, ValueError):
        return str(val)

for prompt in sorted(data['prompt'].dropna().unique()):
    plt.figure(figsize=(8, 5))
    mp = mpc_ps[mpc_ps['prompt'] == prompt]
    for its in sorted(mp['its'].unique()):
        subset = mp[mp['its'] == its]
        mean_subset = (
            subset.groupby('rho_str', dropna=False)
            .agg(
                style_loss_mean=('style_loss_mean', 'mean'),
                clip_score_mean=('clip_score_mean', 'mean'),
                dino_style_cosine_mean=('dino_style_cosine_mean', 'mean'),
                lpips_to_original_mean=('lpips_to_original_mean', 'mean'),
            )
            .reset_index()
        )
        label = f'MPC its={int(its)} (mean)'
        plt.scatter(mean_subset['clip_score_mean'], mean_subset['style_loss_mean'], label=label)
        for _, row in mean_subset.iterrows():
            plt.annotate(
                f"rho={row['rho_str']}",
                (row['clip_score_mean'], row['style_loss_mean']),
                textcoords='offset points',
                xytext=(4, 4),
                fontsize=8,
            )
    fp = flow_ps[flow_ps['prompt'] == prompt]
    for its in sorted(fp['its'].unique()):
        subset = fp[fp['its'] == its]
        mean_subset = (
            subset.groupby('lr', dropna=False)
            .agg(
                style_loss_mean=('style_loss_mean', 'mean'),
                clip_score_mean=('clip_score_mean', 'mean'),
                dino_style_cosine_mean=('dino_style_cosine_mean', 'mean'),
                lpips_to_original_mean=('lpips_to_original_mean', 'mean'),
            )
            .reset_index()
        )
        mean_subset = mean_subset.copy()
        mean_subset['lr_label'] = mean_subset['lr'].apply(format_lr)
        label = f'FlowChef its={int(its)} (mean)'
        plt.scatter(mean_subset['clip_score_mean'], mean_subset['style_loss_mean'], label=label, marker='x')
        for _, row in mean_subset.iterrows():
            plt.annotate(
                f"lr={row['lr_label']}",
                (row['clip_score_mean'], row['style_loss_mean']),
                textcoords='offset points',
                xytext=(4, 4),
                fontsize=8,
            )
    plt.xlabel('Average CLIP agreement')
    plt.ylabel('Average style loss')
    plt.title(f'Per-prompt (mean): {prompt}')
    plt.grid(True, alpha=0.2)
    plt.legend()
    plt.show()

for style in sorted(data['style'].dropna().unique()):
    plt.figure(figsize=(8, 5))
    mp = mpc_ps[mpc_ps['style'] == style]
    for its in sorted(mp['its'].unique()):
        subset = mp[mp['its'] == its]
        mean_subset = (
            subset.groupby('rho_str', dropna=False)
            .agg(
                style_loss_mean=('style_loss_mean', 'mean'),
                clip_score_mean=('clip_score_mean', 'mean'),
                dino_style_cosine_mean=('dino_style_cosine_mean', 'mean'),
                lpips_to_original_mean=('lpips_to_original_mean', 'mean'),
            )
            .reset_index()
        )
        label = f'MPC its={int(its)} (mean)'
        plt.scatter(mean_subset['clip_score_mean'], mean_subset['style_loss_mean'], label=label)
        for _, row in mean_subset.iterrows():
            plt.annotate(
                f"rho={row['rho_str']}",
                (row['clip_score_mean'], row['style_loss_mean']),
                textcoords='offset points',
                xytext=(4, 4),
                fontsize=8,
            )
    fp = flow_ps[flow_ps['style'] == style]
    for its in sorted(fp['its'].unique()):
        subset = fp[fp['its'] == its]
        mean_subset = (
            subset.groupby('lr', dropna=False)
            .agg(
                style_loss_mean=('style_loss_mean', 'mean'),
                clip_score_mean=('clip_score_mean', 'mean'),
                dino_style_cosine_mean=('dino_style_cosine_mean', 'mean'),
                lpips_to_original_mean=('lpips_to_original_mean', 'mean'),
            )
            .reset_index()
        )
        mean_subset = mean_subset.copy()
        mean_subset['lr_label'] = mean_subset['lr'].apply(format_lr)
        label = f'FlowChef its={int(its)} (mean)'
        plt.scatter(mean_subset['clip_score_mean'], mean_subset['style_loss_mean'], label=label, marker='x')
        for _, row in mean_subset.iterrows():
            plt.annotate(
                f"lr={row['lr_label']}",
                (row['clip_score_mean'], row['style_loss_mean']),
                textcoords='offset points',
                xytext=(4, 4),
                fontsize=8,
            )
    plt.xlabel('Average CLIP agreement')
    plt.ylabel('Average style loss')
    plt.title(f'Per-style (mean): {style}')
    plt.grid(True, alpha=0.2)
    plt.legend()
    plt.show()





In [ ]:
import matplotlib.image as mpimg
from pathlib import Path

# Variables to select specific prompt and style
prompts = [
    "a cat",
    "a landscape",
    "a portrait",
    "a dog",
    "two lizards wearing fedoras"
    ]

styles = [
    "style_images/1.png",
    "style_images/bijiasuo.jpeg",
    "style_images/bw.jpg",
    "style_images/jojo.jpeg",
    "style_images/nahan.jpeg",
    "style_images/tan.jpg",
    "style_images/xiangrikui.jpg",
    "style_images/xing.jpg",
    "style_images/xingkong.jpg"
    ]

# 3,4 ; 3,2 ; 3;7 

# 4; 8 is a good choice
# 3;7 fairly good too
# 0;0 excellent too, but missing one lr I think
# 4 ; 4 not bad. 3 ;4 shows FlowChef failing well

full_prompt = prompts[0]
selected_prompt = ''.join(full_prompt.split())[:8]  # e.g., 'acat', 'adog'
full_style = styles[0]
full_style = full_style.split('/')[-1]   # e.g., '1', 'bijiasuo'
selected_style = full_style.split('.')[0]
selected_style_suffix = full_style.split('.')[-1]
selected_its = 20

# Expose which lrs/rhos to include (set to None to include all)
#selected_lrs = [0.0005, 0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5]
#selected_rhos = [1024.0, 512.0, 256.0, 128.0, 64.0, 32.0, 16.0, 8.0, 4.0, 0.0]
selected_lrs = [0.0025, 0.01, 0.025, 0.05]
selected_rhos = [512.0, 64.0, 16.0,  4.0]

#selected_lrs = [0.001]

prompt_key = str(selected_prompt).lower()
style_key = str(selected_style).lower()

plot_data = data.copy()
plot_data = plot_data.dropna(subset=['path', 'prompt', 'style', 'method'])
plot_data['prompt_key'] = plot_data['prompt'].astype(str).str.lower()
plot_data['style_key'] = plot_data['style'].astype(str).str.lower()
plot_data = plot_data[(plot_data['prompt_key'] == prompt_key) & (plot_data['style_key'] == style_key)]

# Original image: MPC rho=na (fallback to any rho=na)
original_row = plot_data[(plot_data['method'] == 'mpc') & (plot_data['rho'].astype(str).str.lower() == 'na')]
if original_row.empty:
    original_row = plot_data[plot_data['rho'].astype(str).str.lower() == 'na']
if not original_row.empty:
    original_row = original_row.sort_values(['its', 'method']).head(1)
original_img_path = None
if not original_row.empty:
    original_img_path = original_row.iloc[0]['path'].replace('.csv', '.png')

# Style image path (from style_images)
style_img_path = repo_root / 'style_images' / f"{selected_style}.{selected_style_suffix}"
if not style_img_path.exists():
    style_img_path = None

# Filter to the selected its for method images
method_data = plot_data[plot_data['its'] == selected_its]

flow_plot = method_data[method_data['method'] == 'flowchef'].copy()
mpc_plot = method_data[method_data['method'] == 'mpc'].copy()

flow_plot['lr_val'] = pd.to_numeric(flow_plot['lr'], errors='coerce')
mpc_plot['rho_val'] = pd.to_numeric(mpc_plot['rho'], errors='coerce')

if selected_lrs is not None:
    flow_plot = flow_plot[flow_plot['lr_val'].isin(selected_lrs)]
if selected_rhos is not None:
    mpc_plot = mpc_plot[mpc_plot['rho_val'].isin(selected_rhos)]

# FlowChef: low -> high lr
flow_plot = flow_plot.sort_values(['lr_val'])
# MPC: high -> low rho
mpc_plot = mpc_plot.sort_values(['rho_val'], ascending=False)

flow_plot['img_path'] = flow_plot['path'].str.replace('.csv', '.png', regex=False)
mpc_plot['img_path'] = mpc_plot['path'].str.replace('.csv', '.png', regex=False)

# Avoid duplicating original if it happens to be in FlowChef/MPC lists
if original_img_path:
    flow_plot = flow_plot[flow_plot['img_path'] != original_img_path]
    mpc_plot = mpc_plot[mpc_plot['img_path'] != original_img_path]

num_flow = len(flow_plot)
num_mpc = len(mpc_plot)
num_cols = max(1 + num_flow, 1 + num_mpc, 2)

fig, axes = plt.subplots(2, num_cols, figsize=(4 * num_cols, 8))
font_size = 14
if num_cols == 1:
    axes = axes.reshape(2, 1)

# Helper to show an image with a fallback
def show_image(ax, img_path, title):
    try:
      img = mpimg.imread(img_path)
      ax.imshow(img)
      ax.set_title(title, fontsize=font_size)
    except FileNotFoundError:
      ax.text(0.5, 0.5, 'Image not found', ha='center', va='center')
      ax.set_title(title, fontsize=font_size)
    ax.axis('off')

# Original image (top row, col 0)
if original_img_path is not None:
    show_image(axes[0, 0], original_img_path, f'"{full_prompt}"')
else:
    axes[0, 0].text(0.5, 0.5, 'Original not found', ha='center', va='center')
    axes[0, 0].set_title('Original (MPC rho=na)', fontsize=font_size)
    axes[0, 0].axis('off')

# Style image (bottom row, col 0)
if style_img_path is not None:
    show_image(axes[1, 0], str(style_img_path), f"Style image")
else:
    axes[1, 0].text(0.5, 0.5, 'Style not found', ha='center', va='center')
    axes[1, 0].set_title(f"Style: {selected_style}", fontsize=font_size)
    axes[1, 0].axis('off')

# FlowChef images in top row (low -> high lr), starting after column 0
for i, row in enumerate(flow_plot.itertuples(), start=1):
    show_image(axes[0, i], row.img_path, f"lr={row.lr}")
for i in range(1 + num_flow, num_cols):
    axes[0, i].axis('off')

# MPC images in second row (high -> low rho), starting after column 0
for i, row in enumerate(mpc_plot.itertuples(), start=1):
    show_image(axes[1, i], row.img_path, f"rho={row.rho}")
for i in range(1 + num_mpc, num_cols):
    axes[1, i].axis('off')

# Add extra space and a black bar between column 0 and 1
plt.subplots_adjust(left=0.02, right=0.98, top=0.98, bottom=0.02, wspace=0.0, hspace=0.0)
if num_cols > 1:
    left_edge = axes[0, 0].get_position().x1
    right_edge = axes[0, 1].get_position().x0
    mid_x = (left_edge + right_edge) / 2
    bar_width = min(0.01, max(0.002, right_edge - left_edge - 0.002))
    bar_x = mid_x - bar_width / 2
    bar = plt.Rectangle((bar_x, 0.05), bar_width, 0.9, transform=fig.transFigure, color='black', zorder=5)
    fig.add_artist(bar)
else:
    mid_x = 0.02

# Row labels between column 0 and 1
fig.text(mid_x, 0.73, 'FlowChef', rotation=90, va='bottom', ha='center', color='white', zorder=6, fontsize=font_size)
fig.text(mid_x, 0.27, 'MPC-Flow', rotation=90, va='bottom', ha='center', color='white', zorder=6, fontsize=font_size)


# Save figure as SVG
out_path = analysis_dir / "sweep_analysis_grid.svg"
fig.savefig(out_path, format="svg")
print(f"Saved: {out_path}")
plt.show()


In [ ]:
import matplotlib.image as mpimg
from pathlib import Path

# Variables to select specific prompt and style
prompts = [
    "a cat",
    "a landscape",
    "a portrait",
    "a dog",
    "two lizards wearing fedoras"
    ]

styles = [
    "style_images/1.png",
    "style_images/bijiasuo.jpeg",
    "style_images/bw.jpg",
    "style_images/jojo.jpeg",
    "style_images/nahan.jpeg",
    "style_images/tan.jpg",
    "style_images/xiangrikui.jpg",
    "style_images/xing.jpg",
    "style_images/xingkong.jpg"
    ]

# 3,4 ; 3,2 ; 3;7 

# 4; 8 is a good choice
# 3;7 fairly good too
# 0;0 excellent too, but missing one lr I think
# 4 ; 4 not bad. 3 ;4 shows FlowChef failing well

full_prompt = prompts[0]
selected_prompt = ''.join(full_prompt.split())[:8]  # e.g., 'acat', 'adog'
full_style = styles[0]
full_style = full_style.split('/')[-1]   # e.g., '1', 'bijiasuo'
selected_style = full_style.split('.')[0]
selected_style_suffix = full_style.split('.')[-1]
selected_its = 20

# Expose which lrs/rhos to include (set to None to include all)
#selected_lrs = [0.0005, 0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5]
#selected_rhos = [1024.0, 512.0, 256.0, 128.0, 64.0, 32.0, 16.0, 8.0, 4.0, 0.0]
selected_lrs = [0.0025, 0.01, 0.025, 0.05]
selected_rhos = [512.0, 64.0, 16.0,  4.0]

#selected_lrs = [0.001]

prompt_key = str(selected_prompt).lower()
style_key = str(selected_style).lower()

plot_data = data.copy()
plot_data = plot_data.dropna(subset=['path', 'prompt', 'style', 'method'])
plot_data['prompt_key'] = plot_data['prompt'].astype(str).str.lower()
plot_data['style_key'] = plot_data['style'].astype(str).str.lower()
plot_data = plot_data[(plot_data['prompt_key'] == prompt_key) & (plot_data['style_key'] == style_key)]

# Original image: MPC rho=na (fallback to any rho=na)
original_row = plot_data[(plot_data['method'] == 'mpc') & (plot_data['rho'].astype(str).str.lower() == 'na')]
if original_row.empty:
    original_row = plot_data[plot_data['rho'].astype(str).str.lower() == 'na']
if not original_row.empty:
    original_row = original_row.sort_values(['its', 'method']).head(1)
original_img_path = None
if not original_row.empty:
    original_img_path = original_row.iloc[0]['path'].replace('.csv', '.png')

# Style image path (from style_images)
style_img_path = repo_root / 'style_images' / f"{selected_style}.{selected_style_suffix}"
if not style_img_path.exists():
    style_img_path = None

# Filter to the selected its for method images
method_data = plot_data[plot_data['its'] == selected_its]

flow_plot = method_data[method_data['method'] == 'flowchef'].copy()
mpc_plot = method_data[method_data['method'] == 'mpc'].copy()

flow_plot['lr_val'] = pd.to_numeric(flow_plot['lr'], errors='coerce')
mpc_plot['rho_val'] = pd.to_numeric(mpc_plot['rho'], errors='coerce')

if selected_lrs is not None:
    flow_plot = flow_plot[flow_plot['lr_val'].isin(selected_lrs)]
if selected_rhos is not None:
    mpc_plot = mpc_plot[mpc_plot['rho_val'].isin(selected_rhos)]

# FlowChef: low -> high lr
flow_plot = flow_plot.sort_values(['lr_val'])
# MPC: high -> low rho
mpc_plot = mpc_plot.sort_values(['rho_val'], ascending=False)

flow_plot['img_path'] = flow_plot['path'].str.replace('.csv', '.png', regex=False)
mpc_plot['img_path'] = mpc_plot['path'].str.replace('.csv', '.png', regex=False)

# Avoid duplicating original if it happens to be in FlowChef/MPC lists
if original_img_path:
    flow_plot = flow_plot[flow_plot['img_path'] != original_img_path]
    mpc_plot = mpc_plot[mpc_plot['img_path'] != original_img_path]

num_flow = len(flow_plot)
num_mpc = len(mpc_plot)
num_cols = max(1 + num_flow, 1 + num_mpc, 2)

fig, axes = plt.subplots(2, num_cols, figsize=(4 * num_cols, 8))
font_size = 18
if num_cols == 1:
    axes = axes.reshape(2, 1)

# Tighten spacing between images
plt.subplots_adjust(wspace=0.05, hspace=0.05)

# Add extra space only between column 0 and column 1 without changing image sizes
if num_cols > 1:
    gap_frac = 0.05
    fig_w, fig_h = fig.get_size_inches()
    gap_in = gap_frac * fig_w
    fig_w_new = fig_w + gap_in
    fig.set_size_inches(fig_w_new, fig_h, forward=True)
    for col in range(num_cols):
        for row in range(2):
            ax = axes[row, col]
            pos = ax.get_position()
            x0_in = pos.x0 * fig_w
            x1_in = pos.x1 * fig_w
            y0_in = pos.y0 * fig_h
            y1_in = pos.y1 * fig_h
            if col >= 1:
                x0_in += gap_in
                x1_in += gap_in
            ax.set_position([
                x0_in / fig_w_new,
                y0_in / fig_h,
                (x1_in - x0_in) / fig_w_new,
                (y1_in - y0_in) / fig_h,
            ])


# Helper to show an image with a fallback
def show_image(ax, img_path, title):
    try:
      img = mpimg.imread(img_path)
      ax.imshow(img)
      ax.set_title(title, fontsize=font_size)
    except FileNotFoundError:
      ax.text(0.5, 0.5, 'Image not found', ha='center', va='center', fontsize=font_size)
      ax.set_title(title, fontsize=font_size)
    ax.axis('off')

# Original image (top row, col 0)
if original_img_path is not None:
    show_image(axes[0, 0], original_img_path, f'"{full_prompt}"')
else:
    axes[0, 0].text(0.5, 0.5, 'Original not found', ha='center', va='center', fontsize=font_size)
    axes[0, 0].set_title('Original (MPC rho=na)', fontsize=font_size)
    axes[0, 0].axis('off')

# Style image (bottom row, col 0)
if style_img_path is not None:
    show_image(axes[1, 0], str(style_img_path), f"Style image")
else:
    axes[1, 0].text(0.5, 0.5, 'Style not found', ha='center', va='center', fontsize=font_size)
    axes[1, 0].set_title(f"Style: {selected_style}", fontsize=font_size)
    axes[1, 0].axis('off')

# FlowChef images in top row (low -> high lr), starting after column 0
for i, row in enumerate(flow_plot.itertuples(), start=1):
    show_image(axes[0, i], row.img_path, f"")
for i in range(1 + num_flow, num_cols):
    axes[0, i].axis('off')

# MPC images in second row (high -> low rho), starting after column 0
for i, row in enumerate(mpc_plot.itertuples(), start=1):
    show_image(axes[1, i], row.img_path, "")
for i in range(1 + num_mpc, num_cols):
    axes[1, i].axis('off')

# Row labels between column 0 and 1
if num_cols > 1:
    left_edge = axes[0, 0].get_position().x1
    right_edge = axes[0, 1].get_position().x0
    mid_x = (left_edge + right_edge) / 2
else:
    mid_x = 0.02

fig.text(mid_x, 0.73, 'FlowChef', rotation=90, va='bottom', ha='center', color='black', zorder=6, fontsize=font_size)
fig.text(mid_x, 0.27, 'MPC-Flow', rotation=90, va='bottom', ha='center', color='black', zorder=6, fontsize=font_size)


axes[1,1].set_title("Increased training-free conditioning strength", fontsize=font_size)

# Save figure as SVG
out_path = analysis_dir / "sweep_analysis_grid.svg"
fig.savefig(out_path, format="svg")
print(f"Saved: {out_path}")
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))

mpc_label_used = False
for its in sorted(mpc_group['its'].unique()):
    subset = mpc_group[mpc_group['its'] == its].sort_values('rho_val')
    subset = subset[subset['rho_str'] != 'na']
    if subset.empty:
        continue
    plt.plot(
        subset['dino_style_cosine_mean'],
        1 - subset['lpips_to_original_mean'],
        linestyle=':',
        linewidth=1.2,
        alpha=0.6,
        label='_nolegend_',
    )
    plt.errorbar(
        subset['dino_style_cosine_mean'],
        1 - subset['lpips_to_original_mean'],
        #xerr=subset['dino_style_cosine_std'],
        #yerr=subset['lpips_to_original_std'],
        fmt='o',
        capsize=2,
        label='MPC-Flow' if not mpc_label_used else '_nolegend_',
    )
    mpc_label_used = True

flow_label_used = False
for its in sorted(flow_group['its'].unique()):
    subset = flow_group[flow_group['its'] == its].sort_values('lr')
    if subset.empty:
        continue
    plt.plot(
        subset['dino_style_cosine_mean'],
        1 - subset['lpips_to_original_mean'],
        linestyle=':',
        linewidth=1.2,
        alpha=0.6,
        label='_nolegend_',
    )
    plt.errorbar(
        subset['dino_style_cosine_mean'],
        1 - subset['lpips_to_original_mean'],
        #xerr=subset['dino_style_cosine_std'],
        #yerr=subset['lpips_to_original_std'],
        fmt='x',
        capsize=2,
        label='FlowChef' if not flow_label_used else '_nolegend_',
    )
    flow_label_used = True

orig = mpc_group[mpc_group['rho_str'] == 'na']
if not orig.empty:
    plt.errorbar(
        orig['dino_style_cosine_mean'],
        1 - orig['lpips_to_original_mean'],
        #xerr=orig['dino_style_cosine_std'],
        #yerr=orig['lpips_to_original_std'],
        fmt='s',
        capsize=2,
        label='Original (no style transfer)',
    )

plt.xlabel('Style adherence (DINO cosine similarity to style image)')
plt.ylabel('Content preservation (1 - LPIPS to original)')
plt.title('Style-Content Trade-off Across Hyperparameters')
plt.grid(True, alpha=0.2)
plt.xlim([-0.01, 0.4])
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))

mpc_label_used = False
for its in sorted(mpc_group['its'].unique()):
    subset = mpc_group[mpc_group['its'] == its].sort_values('rho_val')
    subset = subset[subset['rho_str'] != 'na']
    if subset.empty:
        continue
    plt.plot(
        subset['dino_style_cosine_mean'],
        1 - subset['lpips_to_original_mean'],
        linestyle=':',
        linewidth=1.2,
        alpha=0.6,
        label='_nolegend_',
    )
    plt.errorbar(
        subset['dino_style_cosine_mean'],
        1 - subset['lpips_to_original_mean'],
        xerr=subset['dino_style_cosine_std'],
        yerr=subset['lpips_to_original_std'],
        fmt='o',
        capsize=2,
        label='MPC' if not mpc_label_used else '_nolegend_',
    )
    mpc_label_used = True

flow_label_used = False
for its in sorted(flow_group['its'].unique()):
    subset = flow_group[flow_group['its'] == its].sort_values('lr')
    if subset.empty:
        continue
    plt.plot(
        subset['dino_style_cosine_mean'],
        1 - subset['lpips_to_original_mean'],
        linestyle=':',
        linewidth=1.2,
        alpha=0.6,
        label='_nolegend_',
    )
    plt.errorbar(
        subset['dino_style_cosine_mean'],
        1 - subset['lpips_to_original_mean'],
        xerr=subset['dino_style_cosine_std'],
        yerr=subset['lpips_to_original_std'],
        fmt='x',
        capsize=2,
        label='FlowChef' if not flow_label_used else '_nolegend_',
    )
    flow_label_used = True

orig = mpc_group[mpc_group['rho_str'] == 'na']
if not orig.empty:
    plt.errorbar(
        orig['dino_style_cosine_mean'],
        1 - orig['lpips_to_original_mean'],
        xerr=orig['dino_style_cosine_std'],
        yerr=orig['lpips_to_original_std'],
        fmt='s',
        capsize=2,
        label='Original (no style transfer)',
    )

plt.xlabel('Style adherence (DINO cosine similarity to style image)')
plt.ylabel('Content preservation (1 - LPIPS to original)')
plt.title('Style-Content Trade-off Across Hyperparameters')
plt.grid(True, alpha=0.2)
plt.xlim([-0.01, 0.4])
plt.legend()
plt.show()


In [ ]:
# Image grid for style transfer: prompt+style (top), MPC (middle), FlowChef (bottom)
from pathlib import Path
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

# One entry per column (prompt/style pair).
# - prompt: full prompt text (used for label + prompt_key unless overridden)
# - prompt_key: optional override for sweep key (e.g., 'acat')
# - style: style name without extension (e.g., 'xingkong')
# - style_suffix: file extension in style_images (png/jpg/jpeg)
# - its: optimization steps to select
# - mpc_rho: rho value for MPC (None = pick highest rho)
# - flow_lr: lr value for FlowChef (None = pick lowest lr)

MPC_RHO = 8.0
FLOW_LR = 0.025

COLUMNS = [
    dict(prompt='a cat', style='xingkong', style_suffix='jpg', its=20, mpc_rho=MPC_RHO, flow_lr=FLOW_LR),
    dict(prompt='a dog', style='xiangrikui', style_suffix='jpg', its=20, mpc_rho=MPC_RHO, flow_lr=FLOW_LR),
    dict(prompt='a landscape', style='nahan', style_suffix='jpeg', its=20, mpc_rho=MPC_RHO, flow_lr=FLOW_LR),
    #dict(prompt='a portrait', style='1', style_suffix='png', its=20, mpc_rho=MPC_RHO, flow_lr=FLOW_LR),
    dict(prompt='two lizards wearing fedoras', style='bijiasuo', style_suffix='jpeg', its=20, mpc_rho=MPC_RHO, flow_lr=FLOW_LR),
]

def _prompt_key(text):
    return ''.join(text.split()).lower()[:8]

def _style_path(style, suffix=None):
    if suffix:
        p = repo_root / 'style_images' / f'{style}.{suffix}'
        return p if p.exists() else None
    for ext in ['png', 'jpg', 'jpeg']:
        p = repo_root / 'style_images' / f'{style}.{ext}'
        if p.exists():
            return p
    return None

def _pick_row(df, value_col, value, sort_col=None, ascending=True):
    if value is not None:
        df = df[df[value_col] == value]
    if df.empty:
        return None
    if sort_col:
        df = df.sort_values(sort_col, ascending=ascending)
    return df.iloc[0]

def show_image(ax, img_path, title=None, font_size=14):
    try:
        img = mpimg.imread(img_path)
        ax.imshow(img, aspect='auto')
        ax.margins(0)
        if title:
            ax.set_title(title, fontsize=font_size)
    except (FileNotFoundError, TypeError):
        ax.text(0.5, 0.5, 'Image not found', ha='center', va='center', fontsize=font_size)
        if title:
            ax.set_title(title, fontsize=font_size)
    ax.axis('off')

plot_data = data.dropna(subset=['path', 'prompt', 'style', 'method']).copy()
plot_data['prompt_key'] = plot_data['prompt'].astype(str).str.lower()
plot_data['style_key'] = plot_data['style'].astype(str).str.lower()

ncols = len(COLUMNS)
fig = plt.figure(figsize=(4 * ncols, 8))
gs = fig.add_gridspec(3, ncols, height_ratios=[0.5, 1.0, 1.0], hspace=0.08, wspace=0.05)

for col_idx, col in enumerate(COLUMNS):
    prompt_text = col['prompt']
    prompt_key = col.get('prompt_key') or _prompt_key(prompt_text)
    style_key = str(col['style']).lower()
    its = col.get('its', 20)

    subset = plot_data[(plot_data['prompt_key'] == prompt_key) & (plot_data['style_key'] == style_key)]

    # Unconditional prompt image: MPC rho=na (fallback to any rho=na)
    rho_str = subset['rho'].astype(str).str.lower()
    original_row = subset[(subset['method'] == 'mpc') & (rho_str == 'na')]
    if original_row.empty:
        original_row = subset[rho_str == 'na']
    original_img_path = None
    if not original_row.empty:
        original_img_path = original_row.iloc[0]['path'].replace('.csv', '.png')

    style_img_path = _style_path(col['style'], col.get('style_suffix'))

    # Method images (style transfer)
    method_data = subset[subset['its'] == its].copy()
    flow = method_data[method_data['method'] == 'flowchef'].copy()
    mpc = method_data[method_data['method'] == 'mpc'].copy()

    flow['lr_val'] = pd.to_numeric(flow['lr'], errors='coerce')
    mpc['rho_val'] = pd.to_numeric(mpc['rho'], errors='coerce')

    flow_row = _pick_row(flow, 'lr_val', col.get('flow_lr'), sort_col='lr_val', ascending=True)
    mpc_row = _pick_row(mpc, 'rho_val', col.get('mpc_rho'), sort_col='rho_val', ascending=False)

    flow_img_path = flow_row['path'].replace('.csv', '.png') if flow_row is not None else None
    mpc_img_path = mpc_row['path'].replace('.csv', '.png') if mpc_row is not None else None

    # Row 1: prompt + style (smaller)
    subgs = gs[0, col_idx].subgridspec(1, 2, wspace=0.0)
    ax_prompt = fig.add_subplot(subgs[0, 0])
    show_image(ax_prompt, original_img_path, title=f'"{prompt_text}"', font_size=20)
    ax_style = fig.add_subplot(subgs[0, 1])
    show_image(ax_style, str(style_img_path) if style_img_path else None, title='')

    # Row 2: FlowChef
    ax_flow = fig.add_subplot(gs[1, col_idx])
    show_image(ax_flow, flow_img_path, title=None)

    # Row 3: MPC
    ax_mpc = fig.add_subplot(gs[2, col_idx])
    show_image(ax_mpc, mpc_img_path, title=None)

# Row labels on the left
fig.text(0.01, 0.60, 'FlowChef', rotation=90, va='center', ha='left', fontsize=24)
fig.text(0.01, 0.22, 'MPC-Flow', rotation=90, va='center', ha='left', fontsize=24)
plt.subplots_adjust(left=0.04, right=0.99, top=0.98, bottom=0.02)
plt.savefig(analysis_dir / "style_transfer_grid.pdf", format="pdf", bbox_inches='tight')
plt.savefig(analysis_dir / "style_transfer_grid.svg", format="svg", bbox_inches='tight', dpi=300)
plt.show()